In [2]:
# Since "translation" pipeline is removed in v5.4+, 
# we reuse the same AutoModelForSeq2SeqLM approach:

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── helper: drop-in replacement for the old pipeline("translation") call 
def make_translator(model_name):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.eval()

    def translate(text, max_length=200, **gen_kwargs):
        is_batch = isinstance(text, list)
        texts    = text if is_batch else [text]

        inputs = tokenizer(
            texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_length=max_length,
                num_beams=4,
                early_stopping=True,
                **gen_kwargs
            )
        translated = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        results = [{"translation_text": t} for t in translated]
        return results if is_batch else results[0]

    return translate


In [4]:

# ── 1. Helsinki-NLP/opus-mt — fast, specific language pairs ──────────────────
#    Lightweight MarianMT models, one model per language pair
#    Best for: production use, low latency, known fixed language pairs
# ─────────────────────────────────────────────────────────────────────────────

# English to French
translate = make_translator("Helsinki-NLP/opus-mt-en-fr")

texts = [
    "Machine learning is transforming the way we interact with technology.",
    "The quarterly earnings report exceeded all analyst expectations.",
    "Please ensure your passport is valid for at least six months before travel.",
]

results = translate(texts)
for text, result in zip(texts, results):
    print(f"EN : {text}")
    print(f"FR : {result['translation_text']}") #type: ignore
    print()

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

EN : Machine learning is transforming the way we interact with technology.
FR : L'apprentissage automatique transforme la façon dont nous interagissons avec la technologie.

EN : The quarterly earnings report exceeded all analyst expectations.
FR : Le rapport trimestriel sur les gains a dépassé toutes les attentes des analystes.

EN : Please ensure your passport is valid for at least six months before travel.
FR : Veuillez vous assurer que votre passeport est valide pendant au moins six mois avant le voyage.



model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

In [5]:
# English to German
translate = make_translator("Helsinki-NLP/opus-mt-en-de")

texts = [
    "The new software update includes several critical security patches.",
    "Renewable energy sources are becoming increasingly cost-competitive.",
    "Our customer support team is available 24 hours a day, seven days a week.",
]

results = translate(texts)
for text, result in zip(texts, results):
    print(f"EN : {text}")
    print(f"DE : {result['translation_text']}") #type:ignore
    print()


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/298M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

EN : The new software update includes several critical security patches.
DE : Das neue Software-Update enthält mehrere kritische Sicherheitspatches.

EN : Renewable energy sources are becoming increasingly cost-competitive.
DE : Erneuerbare Energiequellen werden zunehmend kostenwettbewerbsfähiger.

EN : Our customer support team is available 24 hours a day, seven days a week.
DE : Unser Kundensupport-Team steht Ihnen 24 Stunden am Tag, sieben Tage die Woche zur Verfügung.



In [6]:

# English to Hindi
translate = make_translator("Helsinki-NLP/opus-mt-en-hi")

texts = [
    "Artificial intelligence is rapidly changing the healthcare industry.",
    "The government announced a new policy for renewable energy subsidies.",
    "Please submit your application before the deadline on Friday.",
]

results = translate(texts)
for text, result in zip(texts, results):
    print(f"EN : {text}")
    print(f"HI : {result['translation_text']}") #type:ignore
    print()

# EN : Artificial intelligence is rapidly changing the healthcare industry.
# HI : कृत्रिम बुद्धिमत्ता तेजी से स्वास्थ्य सेवा उद्योग को बदल रही है।

# EN : The government announced a new policy for renewable energy subsidies.
# HI : सरकार ने नवीकरणीय ऊर्जा सब्सिडी के लिए एक नई नीति की घोषणा की।

# EN : Please submit your application before the deadline on Friday.
# HI : कृपया शुक्रवार की समय सीमा से पहले अपना आवेदन जमा करें।


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

EN : Artificial intelligence is rapidly changing the healthcare industry.
HI : स्वास्थ्य चिकित्सा उद्योग को तेज़ी से बदल रहा है।

EN : The government announced a new policy for renewable energy subsidies.
HI : सरकार ने नवीकृत ऊर्जा के लिए एक नई नीति घोषित की.

EN : Please submit your application before the deadline on Friday.
HI : शुक्रवार को मृत पंक्ति के पहले कृपया अपने अनुप्रयोग को अधीन करें.



In [7]:
# ── 2. facebook/nllb-200-distilled-600M — 200 languages ──────────────────────
#    Single model covering 200 languages using language codes
#    Best for: less common languages, multilingual pipelines, research
# ─────────────────────────────────────────────────────────────────────────────

# NLLB requires forced_bos_token_id to specify the target language
def make_nllb_translator(model_name="facebook/nllb-200-distilled-600M"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.eval()

    def translate(text, src_lang, tgt_lang, max_length=200):
        is_batch = isinstance(text, list)
        texts    = text if is_batch else [text]

        tokenizer.src_lang = src_lang
        inputs = tokenizer(
            texts,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                forced_bos_token_id=forced_bos_token_id,
                max_length=max_length,
                num_beams=4,
                early_stopping=True
            )
        translated = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        results = [{"translation_text": t} for t in translated]
        return results if is_batch else results[0]

    return translate

translate = make_nllb_translator("facebook/nllb-200-distilled-600M")

# same source text translated into multiple languages
source = "Climate change is one of the most pressing challenges facing humanity today."

language_pairs = [
    ("eng_Latn", "fra_Latn", "French"),
    ("eng_Latn", "hin_Deva", "Hindi"),
    ("eng_Latn", "ara_Arab", "Arabic"),
    ("eng_Latn", "zho_Hans", "Chinese (Simplified)"),
    ("eng_Latn", "swh_Latn", "Swahili"),
]

print(f"Source (EN): {source}\n")
for src_lang, tgt_lang, lang_name in language_pairs:
    result = translate(source, src_lang=src_lang, tgt_lang=tgt_lang)
    print(f"{lang_name:25s}: {result['translation_text']}") #type:ignore

# Source (EN): Climate change is one of the most pressing challenges facing humanity today.

# French                   : Le changement climatique est l'un des défis les plus pressants auxquels l'humanité est confrontée aujourd'hui.
# Hindi                    : जलवायु परिवर्तन आज मानवता के सामने सबसे गंभीर चुनौतियों में से एक है।
# Arabic                   : يُعدّ تغير المناخ أحد أكثر التحديات التي تواجه البشرية إلحاحاً اليوم.
# Chinese (Simplified)     : 气候变化是当今人类面临的最紧迫挑战之一。
# Swahili                  : Mabadiliko ya tabianchi ni moja ya changamoto kubwa zaidi zinazokabili ubinadamu leo.


# batch: translate multiple sentences at once
texts = [
    "The meeting has been rescheduled to next Monday at 9am.",
    "Please review the attached document and provide your feedback by Thursday.",
    "The project deadline has been extended by two weeks due to scope changes.",
]

results = translate(texts, src_lang="eng_Latn", tgt_lang="fra_Latn")
for text, result in zip(texts, results):
    print(f"EN : {text}")
    print(f"FR : {result['translation_text']}") #type:ignore
    print()

# EN : The meeting has been rescheduled to next Monday at 9am.
# FR : La réunion a été reportée au lundi prochain à 9h.

# EN : Please review the attached document and provide your feedback by Thursday.
# FR : Veuillez examiner le document ci-joint et fournir vos commentaires d'ici jeudi.

# EN : The project deadline has been extended by two weeks due to scope changes.
# FR : La date limite du projet a été prolongée de deux semaines en raison de changements de portée.


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Source (EN): Climate change is one of the most pressing challenges facing humanity today.

French                   : Le changement climatique est l'un des défis les plus pressants auxquels l'humanité est confrontée aujourd'hui.
Hindi                    : जलवायु परिवर्तन आज मानवता के सामने सबसे महत्वपूर्ण चुनौतियों में से एक है।
Arabic                   : Climate change is one of the most pressing challenges facing humanity today.
Chinese (Simplified)     : 气候变化是今天人类面临的最紧迫的挑战之一.
Swahili                  : Mabadiliko ya hali ya hewa ni mojawapo ya changamoto muhimu zaidi zinazowakabili wanadamu leo.
EN : The meeting has been rescheduled to next Monday at 9am.
FR : La réunion a été reportée au lundi prochain à 9 h.

EN : Please review the attached document and provide your feedback by Thursday.
FR : Veuillez examiner le document joint et fournir vos commentaires d'ici jeudi.

EN : The project deadline has been extended by two weeks due to scope changes.
FR : Le délai du projet a été prol

In [8]:
# batch: translate multiple sentences at once
texts = [
    "The meeting has been rescheduled to next Monday at 9am.",
    "Please review the attached document and provide your feedback by Thursday.",
    "The project deadline has been extended by two weeks due to scope changes.",
]

results = translate(texts, src_lang="eng_Latn", tgt_lang="fra_Latn")
for text, result in zip(texts, results):
    print(f"EN : {text}")
    print(f"FR : {result['translation_text']}") #type:ignore
    print()

# EN : The meeting has been rescheduled to next Monday at 9am.
# FR : La réunion a été reportée au lundi prochain à 9h.

# EN : Please review the attached document and provide your feedback by Thursday.
# FR : Veuillez examiner le document ci-joint et fournir vos commentaires d'ici jeudi.

# EN : The project deadline has been extended by two weeks due to scope changes.
# FR : La date limite du projet a été prolongée de deux semaines en raison de changements de portée.

EN : The meeting has been rescheduled to next Monday at 9am.
FR : La réunion a été reportée au lundi prochain à 9 h.

EN : Please review the attached document and provide your feedback by Thursday.
FR : Veuillez examiner le document joint et fournir vos commentaires d'ici jeudi.

EN : The project deadline has been extended by two weeks due to scope changes.
FR : Le délai du projet a été prolongé de deux semaines en raison de changements de portée.



In [9]:


# ── 3. back-translation with Helsinki-NLP ────────────────────────────────────
#    Translate to target language then back to source to verify quality
#    Best for: data augmentation, quality checking, training data generation
# ─────────────────────────────────────────────────────────────────────────────
en_to_es = make_translator("Helsinki-NLP/opus-mt-en-es")
es_to_en = make_translator("Helsinki-NLP/opus-mt-es-en")

originals = [
    "The board of directors approved the merger after months of negotiation.",
    "Early diagnosis significantly improves the chances of successful treatment.",
    "The new regulation requires all companies to disclose their carbon emissions annually.",
]

print("Back-translation quality check (EN -> ES -> EN):\n")
for original in originals:
    spanish    = en_to_es(original)["translation_text"] #type:ignore
    back       = es_to_en(spanish)["translation_text"] #type:ignore
    print(f"Original  : {original}")
    print(f"Spanish   : {spanish}")
    print(f"Back (EN) : {back}")
    print()

# Original  : The board of directors approved the merger after months of negotiation.
# Spanish   : La junta directiva aprobó la fusión después de meses de negociación.
# Back (EN) : The board of directors approved the merger after months of negotiation.

# Original  : Early diagnosis significantly improves the chances of successful treatment.
# Spanish   : El diagnóstico temprano mejora significativamente las posibilidades de un tratamiento exitoso.
# Back (EN) : Early diagnosis significantly improves the chances of successful treatment.

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Back-translation quality check (EN -> ES -> EN):



model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Original  : The board of directors approved the merger after months of negotiation.
Spanish   : El consejo de administración aprobó la fusión después de meses de negociación.
Back (EN) : The Board of Directors approved the merger after months of negotiation.

Original  : Early diagnosis significantly improves the chances of successful treatment.
Spanish   : El diagnóstico temprano mejora significativamente las posibilidades de un tratamiento exitoso.
Back (EN) : Early diagnosis significantly improves the chances of successful treatment.

Original  : The new regulation requires all companies to disclose their carbon emissions annually.
Spanish   : El nuevo reglamento exige que todas las empresas revelen sus emisiones de carbono anualmente.
Back (EN) : The new regulation requires all companies to disclose their carbon emissions annually.



### **text-generation**

In [ ]:
# **Quick reference:**

# | Model | Size | Best for |
# |---|---|---|
# | `gpt2` | 117M | Learning, fast testing, small prompts |
# | `distilgpt2` | 82M | Fastest, edge/resource-constrained |
# | `gpt2-medium/large/xl` | 345M–1.5B | Progressively better GPT-2 quality |
# | `Qwen/Qwen2.5-1.5B` | 1.5B | High quality without GPU |
# | `meta-llama/Llama-3.2-1B` | 1B | Strong general generation, needs HF token |
# | `mistralai/Mistral-7B-v0.1` | 7B | Best quality, requires GPU |

# | Parameter | Effect |
# |---|---|
# | `max_new_tokens` | How many tokens to generate beyond the prompt |
# | `do_sample=False` | Greedy decoding — deterministic, factual |
# | `temperature` | Higher = more creative, lower = more focused |
# | `top_k` | Sample only from top K most likely tokens |
# | `top_p` | Nucleus sampling — cut off at cumulative probability p |
# | `repetition_penalty` | Values above 1.0 reduce word repetition |
# | `return_full_text=False` | Return only generated text, not the prompt |
# | `num_return_sequences` | Number of independent completions to generate |

In [12]:
from transformers import pipeline

# ── 1. gpt2 ──────────────────────────────────────────────────────────────────
#    Classic small model, great for learning and experimentation
#    Best for: understanding generation basics, fast local testing
# ─────────────────────────────────────────────────────────────────────────────

pipe = pipeline("text-generation", model="gpt2")

# basic continuation
result = pipe(
    "The history of the internet began in",
    max_new_tokens=80,       # correct parameter name
    do_sample=False,
    return_full_text=False   # remove temperature when do_sample=False
)

print(result[0]["generated_text"])

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


 the early 1990s when the internet was first created. The internet was created by a group of people who wanted to make a better world. The internet was created by people who wanted to make a better world.

The internet was created by a group of people who wanted to make a better world.

The internet was created by a group of people who wanted to make a better world.


In [13]:

# multiple completions with sampling
results = pipe(
    "Scientists have recently discovered that",
    max_new_tokens=60,
    num_return_sequences=3,
    do_sample=True,
    temperature=0.9,
    top_k=50,
    top_p=0.95,
    return_full_text=False
)
print("3 different completions:\n")
for i, r in enumerate(results, 1):
    print(f"  [{i}] {r['generated_text'].strip()}")
    print()

# [1] a new species of deep-sea fish off the coast of New Zealand that can
#     survive at depths of over 3,000 metres...
# [2] the human brain forms new neural connections throughout adulthood,
#     challenging the long-held belief that neuroplasticity declines after...
# [3] a chemical compound found in green tea may slow the progression of
#     certain neurodegenerative diseases including Parkinson's...



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


3 different completions:

  [1] the brain activity that makes a person remember a certain story is actually not part of their memory. Research published in the February 18 issue of the Journal of Psychophysiology finds evidence that brain regions that are involved in memory recall are indeed involved in memory formation.

The scientists found that during memory recall

  [2] they don't only produce hydrogen and oxygen but also make it radioactive. The scientists believe this would make it more attractive for them to use the gas to build more weapons.


But the company has also added a new element in its arsenal – iron oxide. It has developed a method for making it

  [3] human genes are responsible for a variety of different kinds of behavior in birds.

The discovery of the new gene sets off several major studies on the genetics of bird behavior. First, researchers discovered that the genetic mechanism behind flight has been altered at different levels in birds. Some of the genes respo

In [14]:
# batch: multiple prompts at once
prompts = [
    "The stock market crashed when",
    "In the year 2045, robots will",
    "The greatest challenge facing modern medicine is",
]

results = pipe(
    prompts,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    return_full_text=False
)

for prompt, result in zip(prompts, results):
    print(f"Prompt : {prompt}")
    print(f"Output : {result[0]['generated_text'].strip()}")
    print()


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt : The stock market crashed when
Output : President Obama won the presidential election. The stock market crashed when Wall Street crashed.

This year's stock market is the one big crash of the summer on the eve of Election Day. A big crash is a big mess. And it is no

Prompt : In the year 2045, robots will
Output : be able to do things such as make medical treatments, monitor patients and more.

But what about that future that will make human and machine collaboration so much more feasible?

"When you have a human robot out there doing things that humans

Prompt : The greatest challenge facing modern medicine is
Output : making sure that we develop more effective treatments. This requires the establishment of scientific, clinical, scientific and technological breakthroughs. To achieve this, we need to find a way to reduce costs.

These are just some of the challenges faced by doctors



In [15]:
# result is a list of dicts, so inspect the first element
print(result)
# [{'generated_text': '...'}]

# keys of the dict
print(result[0].keys())
# dict_keys(['generated_text'])

# each key and value
for k, v in result[0].items():
    print(f"{k}: {v}")
# generated_text: the 1960s, when the United States...

# if you want to inspect the type and structure
print(type(result))        # <class 'list'>
print(type(result[0]))     # <class 'dict'>
print(len(result))         # 1  (or more if num_return_sequences > 1)

[{'generated_text': ' making sure that we develop more effective treatments. This requires the establishment of scientific, clinical, scientific and technological breakthroughs. To achieve this, we need to find a way to reduce costs.\n\nThese are just some of the challenges faced by doctors'}]
dict_keys(['generated_text'])
generated_text:  making sure that we develop more effective treatments. This requires the establishment of scientific, clinical, scientific and technological breakthroughs. To achieve this, we need to find a way to reduce costs.

These are just some of the challenges faced by doctors
<class 'list'>
<class 'dict'>
1


In [16]:
# ── 2. distilgpt2 ────────────────────────────────────────────────────────────
#    Smallest and fastest GPT-2 variant, 40 percent fewer parameters
#    Best for: high-throughput, edge deployment, resource-constrained systems
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("text-generation", model="distilgpt2")

# creative writing
prompts = [
    "Deep in the forest, an old man discovered",
    "The last astronaut on the space station looked out the window and saw",
    "She opened the letter and read the words that would change everything:",
]

for prompt in prompts:
    result = pipe(
        prompt,
        max_new_tokens=70,
        do_sample=True,
        temperature=0.85,
        top_p=0.92,
        repetition_penalty=1.3,
        return_full_text=False
    )
    print(f"Prompt : {prompt}")
    print(f"Output : {result[0]['generated_text'].strip()}")
    print()

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt : Deep in the forest, an old man discovered
Output : something very strange inside him. He had been hiding there for long without any notice to anyone and he was not sure what happened that night but it seemed like a lot of his life would be lost if you didn't keep silent about this mysterious figure."
"I don´t want to lose anything," said Olukana while looking at her



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt : The last astronaut on the space station looked out the window and saw
Output : a man standing next to him.
“This is what you see, my favorite photo of any ISS crew member in history? I thought that would be great." ‟"You can take pictures from all this while sitting there or watching movies — so it's just an excuse for getting up here … maybe one more time before going down somewhere

Prompt : She opened the letter and read the words that would change everything:
Output : “The world is a lot more complicated than it used to be.”
But for those unfamiliar with China, all these things don't mean nothing in any way—it's not something we've seen from America since President Obama took office last year or even just one day before this election cycle — but if you want to know what I



In [17]:
# ── 3. Qwen/Qwen2.5-1.5B ─────────────────────────────────────────────────────
#    Efficient 1.5B model from Alibaba, much stronger than GPT-2 at same size
#    Best for: higher quality output without requiring a GPU
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("text-generation",
                model="Qwen/Qwen2.5-1.5B",
                device_map="auto")

# factual continuation
prompts = [
    "The key differences between supervised and unsupervised learning are",
    "To build a REST API in Python using FastAPI, you need to",
    "The main causes of the 2008 financial crisis were",
]

for prompt in prompts:
    result = pipe(
        prompt,
        max_new_tokens=120,
        do_sample=False,           # greedy for factual content
        repetition_penalty=1.2,
        return_full_text=False
    )
    print(f"Prompt : {prompt}")
    print(f"Output : {result[0]['generated_text'].strip()}")
    print()

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


Prompt : The key differences between supervised and unsupervised learning are
Output : as follows:
A. Supervised learning requires the use of labeled data, while unsupervised learning does not.
B. In supervised learning tasks, there is a need to predict continuous values; in contrast, for unsupervised learning problems, it's necessary to identify discrete categories or clusters from unlabeled datasets.
C. The output results produced by supervised learning can be directly used without further processing (e.g., classification), whereas this isn't true for unsupervised learning outputs which often require additional steps such as feature extraction before they can be applied effectively.

Please select all correct descriptions above that accurately

Prompt : To build a REST API in Python using FastAPI, you need to
Output : follow these steps:

1. Install the required packages: `fastapi`, `uvicorn` and `pydantic`.
2. Create an application class that inherits from `BaseApplication`. This wi

In [18]:
# ── 4. generation parameters comparison ──────────────────────────────────────
#    Shows how temperature and sampling strategy affect output style
#    Same prompt, four different parameter configurations
# ─────────────────────────────────────────────────────────────────────────────
pipe = pipeline("text-generation", model="gpt2")

prompt = "The future of renewable energy looks"

configs = [
    {
        "label":       "Greedy (deterministic)",
        "do_sample":   False,
        "temperature": 1.0,
    },
    {
        "label":       "Low temperature (focused)",
        "do_sample":   True,
        "temperature": 0.3,
        "top_k":       40,
    },
    {
        "label":       "Medium temperature (balanced)",
        "do_sample":   True,
        "temperature": 0.8,
        "top_p":       0.92,
    },
    {
        "label":       "High temperature (creative)",
        "do_sample":   True,
        "temperature": 1.4,
        "top_k":       100,
        "top_p":       0.98,
    },
]

print(f"Prompt: {prompt}\n")
for config in configs:
    label = config.pop("label")
    result = pipe(
        prompt,
        max_new_tokens=60,
        return_full_text=False,
        repetition_penalty=1.2,
        **config
    )
    print(f"  [{label}]")
    print(f"  {result[0]['generated_text'].strip()}")
    print()

# [Greedy (deterministic)]
#   bright, with solar and wind power expected to account for more than
#   50 percent of global electricity generation by 2035...

# [Low temperature (focused)]
#   very promising, with solar capacity growing faster than any other energy
#   source in history. Battery storage costs have fallen dramatically...

# [Medium temperature (balanced)]
#   increasingly competitive, driven by falling costs and strong government
#   policy support. Offshore wind projects are being developed at a scale
#   that was unimaginable just a decade ago...

# [High temperature (creative)]
#   stranger than anyone predicted — not the clean utopia of magazine covers
#   but something messier, political, full of unexpected winners and losers
#   fighting over land, water, minerals, and grid access...

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Prompt: The future of renewable energy looks

  [Greedy (deterministic)]
  bright for the United States, but it's not going to be easy.

  [Low temperature (focused)]
  bright.



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


  [Medium temperature (balanced)]
  bleak.

  [High temperature (creative)]
  bright in its own right, according to one economist
 - David Rosebud (author), The National Interest : Renewableenergy development has evolved profoundly thanks with enormous financial aid... we are on the road towards our natural means – human capital — for some 45 years out...

